# Project D — Task **(b)**: Most specific / leaf-level classification

**Assignment:** assign articles to the **most specific** topic levels possible (full hierarchy, leaf-level evaluation).

This notebook loads the same pool and tree, then:
- **Leaf-level metrics** (test): multilabel leaf F1, path-to-gold branching recall — for **all three** linear baselines with **H1-only** training (same as the exploratory leaf section in the combined demo).
- **Full tree:** train **every** branching edge in the taxonomy (`binary_edge_specs`), with **verbose progress** per edge, then overall **`ft_*`** metrics and leaf scores on the **test** split — again for all three models.

For **task (a)** (four top-level topics only), see **`hierarchical_1a.ipynb`**.

**Training data per edge:** `train_full_tree_and_summarize` / `fit_all_binary_edges` use **local** training by default (`restrict_to_parent_subtree=True`): each edge trains only on articles whose gold intersects the **parent** subtree, so past Root the verbose `fit: n=...` counts are **smaller** than the full training split. Pass `restrict_to_parent_subtree=False` for the legacy **global** baseline (all documents on every edge).

**Working directory:** `Homework 4`. **Warning:** full-tree training is **slow** (many edges × 3 models).

In [1]:
from pathlib import Path

from topic_hierarchy import binary_edge_specs, load_topic_tree, summary


def homework4_base() -> Path:
    """Resolve Homework 4 folder (contains topics.csv)."""
    here = Path.cwd().resolve()
    if (here / "topics.csv").exists():
        return here
    for p in [here, *here.parents]:
        if (p / "topics.csv").exists():
            return p
    raise FileNotFoundError("topics.csv not found — set cwd to Homework 4 or edit BASE")


BASE = homework4_base()
BASE

PosixPath('/Users/nikhileshbelulkar/Documents/HW-Spring-2026/Financial data science and computing/Homework 4')

## Topic tree summary

In [2]:
tree = load_topic_tree(str(BASE / "topics.csv"))
s = summary(tree)
for k, v in sorted(s.items()):
    print(f"  {k}: {v}")

  max_branching_factor: 23
  max_depth_from_root: 5
  n_binary_edge_classifiers: 103
  n_branching_nodes: 22
  n_leaves: 82
  n_local_classifiers: 22
  n_nodes: 117
  traversal_root: Root


## Binary edge inventory (sample)

Each row is one trainable **binary** model: membership in the child’s subtree. Total count is **`n_binary_edge_classifiers`** in the summary above.

In [3]:
edges = binary_edge_specs(tree)
print(f"Total binary edges: {len(edges)}\n")
for sp in edges[:10]:
    print(f"  {sp.parent!r} -> {sp.child!r}  (depth {sp.depth})")

Total binary edges: 103

  'Root' -> 'CCAT'  (depth 0)
  'Root' -> 'ECAT'  (depth 0)
  'Root' -> 'GCAT'  (depth 0)
  'Root' -> 'MCAT'  (depth 0)
  'CCAT' -> 'C1'  (depth 1)
  'CCAT' -> 'C2'  (depth 1)
  'CCAT' -> 'C3'  (depth 1)
  'CCAT' -> 'C4'  (depth 1)
  'ECAT' -> 'E1'  (depth 1)
  'ECAT' -> 'E2'  (depth 1)


In [4]:
from hierarchical_training_data import make_multilabel_binary_pool_data

mldata = make_multilabel_binary_pool_data(base_path=str(BASE))
print("train", len(mldata.train_ids()), "test", len(mldata.test_ids()))

train 14465 test 3617


## Full tree: train all binary edges + overall / leaf metrics (test)

**Training:** every branching edge in `topics.csv` (`binary_edge_specs`), one model per `parent → child`, same pool and gold as elsewhere. Skips an edge if the **train** split has only one class. Progress is printed as `[i/N] parent → child (depth d)` so you can see where a long run is.

**Overall edge metrics (test, `ft_*`):**

- **`ft_pooled_micro_f1`**: one F1 over **all** (document × edge) pairs — micro-averaged over every binary decision.
- **`ft_macro_f1`**: unweighted mean of per-edge F1 (each edge counts equally).
- **`ft_pos_weighted_f1`**: mean of per-edge F1 weighted by the number of **positive** gold labels on that edge in the test split (edges with more positives get more weight).

**Leaf:** same `leaf_*` and `path_*` metrics as the earlier leaf section, now with a **fully trained** router so routing can reach deep leaves.

**Models:** same three linear factories (`LinearSVC`, `GLMNet_elasticnet`, `MaxEnt_L2`). **This is very slow** (many edges × TF-IDF fits × 3 models); run when ready.

In [5]:
import pickle
import time

from hierarchical_summary_metrics import (
    linear_model_factories,
    train_full_tree_and_summarize,
)

# Full taxonomy: all binary edges, all three models, test metrics + leaf/path.
factories = linear_model_factories(max_features=8000)
full_tree_rows = []
for mi, (model_name, factory) in enumerate(factories.items(), start=1):
    print("\n" + "=" * 72, flush=True)
    print(f"MODEL [{mi}/{len(factories)}] {model_name}", flush=True)
    print("=" * 72 + "\n", flush=True)
    t_wall = time.perf_counter()
    _, row = train_full_tree_and_summarize(
        model_name,
        tree,
        mldata,
        factory,
        split="test",
        verbose=True,
    )
    print(
        f"\n>>> {model_name} finished in {time.perf_counter() - t_wall:.1f}s wall time\n",
        flush=True,
    )
    full_tree_rows.append(row)

FULL_TREE_ROWS_PKL = BASE / "full_tree_rows.pkl"
with open(FULL_TREE_ROWS_PKL, "wb") as f:
    pickle.dump(full_tree_rows, f)
print(f"Saved {len(full_tree_rows)} summary row(s) to {FULL_TREE_ROWS_PKL}")


MODEL [1/3] LinearSVC

[1/103] Root → CCAT  (depth 0)
    fit: n=14465  positives=6540
[2/103] Root → ECAT  (depth 0)
    fit: n=14465  positives=4496
[3/103] Root → GCAT  (depth 0)
    fit: n=14465  positives=6402
[4/103] Root → MCAT  (depth 0)
    fit: n=14465  positives=2035
[5/103] CCAT → C1  (depth 1)
    fit: n=6540  positives=4025
[6/103] CCAT → C2  (depth 1)
    fit: n=6540  positives=1288
[7/103] CCAT → C3  (depth 1)
    fit: n=6540  positives=2054
[8/103] CCAT → C4  (depth 1)
    fit: n=6540  positives=806
[9/103] ECAT → E1  (depth 1)
    fit: n=4496  positives=2084
[10/103] ECAT → E2  (depth 1)
    fit: n=4496  positives=765
[11/103] ECAT → E3  (depth 1)
    fit: n=4496  positives=345
[12/103] ECAT → E4  (depth 1)
    fit: n=4496  positives=753
[13/103] ECAT → E5  (depth 1)
    fit: n=4496  positives=872
[14/103] ECAT → E6  (depth 1)
    fit: n=4496  positives=149
[15/103] ECAT → E7  (depth 1)
    fit: n=4496  positives=168
[16/103] GCAT → G1  (depth 1)
    fit: n=6402  pos

### Comparison table (no retrain)

Run the next cell to format `full_tree_rows` (in memory or loaded from `full_tree_rows.pkl`). Adjust **`ACTIVE_GROUPS`**, **`ROUND_DIGITS`**, or **`LOAD_FROM_PICKLE`** without re-running the full-tree training cell.

In [6]:
import pickle
from IPython.display import Markdown, display

import numpy as np
import pandas as pd

from hierarchical_summary_metrics import comparison_table

# --- knobs (edit without retraining) ---
LOAD_FROM_PICKLE = False
FULL_TREE_ROWS_PKL = BASE / "full_tree_rows.pkl"
ROUND_DIGITS = 4
# Subset of: "edge", "leaf_path", "fit"
ACTIVE_GROUPS = ["edge", "leaf_path", "fit"]
HIGHLIGHT_BEST = True

METRIC_GROUPS = {
    "edge": [
        "ft_pooled_micro_f1",
        "ft_pooled_micro_precision",
        "ft_pooled_micro_recall",
        "ft_pooled_micro_accuracy",
        "ft_macro_f1",
        "ft_macro_precision",
        "ft_macro_recall",
        "ft_pos_weighted_f1",
        "ft_pos_weighted_precision",
        "ft_pos_weighted_recall",
        "ft_n_edges_scored",
    ],
    "leaf_path": None,
    "fit": ["fit_n_specs", "fit_n_edges_trained", "fit_n_skipped_single_class"],
}

_SKIP_HIGHLIGHT = {
    "ft_n_edges_scored",
    "fit_n_specs",
    "fit_n_edges_trained",
    "fit_n_skipped_single_class",
    "n_path_gold_branches",
    "n_path_gold_branches_correct",
    "ft_pooled_micro_accuracy",
}


def _group_columns(df: pd.DataFrame, group_name: str) -> list:
    cols = list(df.columns)
    if group_name == "edge":
        order = METRIC_GROUPS["edge"]
        return [c for c in order if c in cols]
    if group_name == "leaf_path":
        return [c for c in cols if c.startswith("leaf_") or c.startswith("path_")]
    if group_name == "fit":
        order = METRIC_GROUPS["fit"]
        return [c for c in order if c in cols]
    return []


def _round_numeric(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    num = list(out.select_dtypes(include=["number"]).columns)
    if num:
        out[num] = out[num].round(ROUND_DIGITS)
    return out


def _maybe_style(df: pd.DataFrame) -> object:
    if not HIGHLIGHT_BEST or df.empty:
        return df
    cand = [
        c
        for c in df.columns
        if c != "model"
        and c not in _SKIP_HIGHLIGHT
        and pd.api.types.is_numeric_dtype(df[c])
    ]
    if not cand:
        return df
    return df.style.highlight_max(axis=0, subset=cand)


if LOAD_FROM_PICKLE:
    with open(FULL_TREE_ROWS_PKL, "rb") as f:
        full_tree_rows = pickle.load(f)
elif "full_tree_rows" not in globals():
    raise RuntimeError(
        "full_tree_rows is not defined — run the training cell above, or set LOAD_FROM_PICKLE = True"
    )

full_tree_df = comparison_table(full_tree_rows)

cols_order = list(full_tree_df.columns)
if "model" in cols_order:
    cols_order.remove("model")
    full_tree_df = full_tree_df[["model"] + cols_order]

numeric_cols = [
    c
    for c in full_tree_df.columns
    if c != "model" and pd.api.types.is_numeric_dtype(full_tree_df[c])
]
scalar_df = (
    full_tree_df[["model"] + numeric_cols]
    if "model" in full_tree_df.columns
    else full_tree_df[numeric_cols]
)
full_rounded = _round_numeric(scalar_df)
display(
    Markdown(
        "#### Full table (numeric scalars only; pooled / per-edge confusion objects below)"
    )
)
display(full_rounded)

display(Markdown("#### By metric group"))
for g in ACTIVE_GROUPS:
    gc = _group_columns(full_tree_df, g)
    if not gc:
        continue
    base = ["model"] if "model" in full_tree_df.columns else []
    sub = full_tree_df[base + gc]
    display(Markdown(f"**{g}**"))
    display(_maybe_style(_round_numeric(sub)))

display(Markdown("#### Pooled binary confusion matrix"))
display(
    Markdown(
        "All branching edges × documents: rows = true class, columns = predicted "
        "(`sklearn.metrics.confusion_matrix`, `labels=[0,1]`)."
    )
)
for row in full_tree_rows:
    name = row.get("model", "?")
    cm = row.get("ft_confusion_matrix_pooled")
    if cm is None:
        continue
    cm = np.asarray(cm)
    display(Markdown(f"**{name}**"))
    display(
        pd.DataFrame(
            cm,
            index=["true 0", "true 1"],
            columns=["pred 0", "pred 1"],
        )
    )

display(Markdown("#### Per-edge confusion matrices"))
display(
    Markdown(
        "Each value is a 2×2 `ndarray` for one `(parent, child)` edge. "
        "Access programmatically as `row['ft_per_edge_confusion']` on a summary dict."
    )
)
for row in full_tree_rows:
    name = row.get("model", "?")
    ped = row.get("ft_per_edge_confusion")
    if not ped:
        continue
    display(Markdown(f"**{name}**: {len(ped)} edges with matrices"))

#### Full table (numeric scalars only; pooled / per-edge confusion objects below)

,model,ft_pooled_micro_f1,ft_macro_f1,ft_pos_weighted_f1,ft_n_edges_scored,ft_pooled_micro_precision,ft_pooled_micro_recall,ft_pooled_micro_accuracy,ft_macro_precision,ft_macro_recall,...,leaf_micro_f1,leaf_macro_f1,leaf_weighted_f1,leaf_samples_f1,path_gold_branch_recall,n_path_gold_branches,n_path_gold_branches_correct,fit_n_specs,fit_n_edges_trained,fit_n_skipped_single_class
0,LinearSVC,0.8351,0.7976,0.8300,99.0,0.8797,0.7947,0.9430,0.8785,0.7417,...,0.6455,0.6233,0.6496,0.5677,0.8004,13631.0,10910.0,103.0,101.0,2.0
1,GLMNet_elasticnet,0.7932,0.6796,0.7728,99.0,0.9047,0.7061,0.9331,0.9059,0.5792,...,0.5510,0.5095,0.5409,0.4418,0.7091,13631.0,9666.0,103.0,101.0,2.0
2,MaxEnt_L2,0.7923,0.6689,0.7693,99.0,0.9096,0.7018,0.9331,0.9159,0.5682,...,0.5412,0.4967,0.5289,0.4294,0.7044,13631.0,9601.0,103.0,101.0,2.0


#### By metric group

**edge**

,model,ft_pooled_micro_f1,ft_pooled_micro_precision,ft_pooled_micro_recall,ft_pooled_micro_accuracy,ft_macro_f1,ft_macro_precision,ft_macro_recall,ft_pos_weighted_f1,ft_pos_weighted_precision,ft_pos_weighted_recall,ft_n_edges_scored
0,LinearSVC,0.835100,0.879700,0.794700,0.943000,0.797600,0.878500,0.741700,0.830000,0.877000,0.794700,99.000000
1,GLMNet_elasticnet,0.793200,0.904700,0.706100,0.933100,0.679600,0.905900,0.579200,0.772800,0.902200,0.706100,99.000000
2,MaxEnt_L2,0.792300,0.909600,0.701800,0.933100,0.668900,0.915900,0.568200,0.769300,0.908400,0.701800,99.000000


...

#### Per-edge confusion matrices

Each value is a 2×2 `ndarray` for one `(parent, child)` edge. Access programmatically as `row['ft_per_edge_confusion']` on a summary dict.

**LinearSVC**: 99 edges with matrices

**GLMNet_elasticnet**: 99 edges with matrices

**MaxEnt_L2**: 99 edges with matrices